# 00 · Creación de la base SQLite

**Proyecto:** Detección de fraude transaccional con SQL, Python y modelado

**Objetivo del notebook:** convertir los CSV del IEEE-CIS en una base SQLite que sirva como fuente única de datos para el resto del proyecto.

**Fuente:** IEEE-CIS Fraud Detection (Vesta Corporation), archivos `train_transaction.csv` y `train_identity.csv`, que se descargan de Kaggle y se ubican en la carpeta `data/`.

In [1]:
from fraude import base_datos as bd, config

# ==========================================================
# Conversión de CSVs del IEEE-CIS a una base de datos SQLite
# ==========================================================
# Los CSV se leen por lotes de 50,000 filas para no cargar los 683 MB de
# transacciones en memoria. Si la base ya existe, solo se verifican los índices.

estado = bd.crear_base()

size_mb = config.BASE.stat().st_size / (1024 * 1024)
print(f"Base de datos {estado}: {config.BASE.name} ({size_mb:.1f} MB)")
print(f"Ubicación: {config.BASE.relative_to(config.RAIZ)}")

Base de datos existente: ieee_fraud.sqlite (385.3 MB)
Ubicación: data/ieee_fraud.sqlite


In [2]:
# ==========================================================
# Reconexión y verificación de tablas
# ==========================================================

conn = bd.conectar()

resumen = bd.resumen_tablas(conn)
print("Tablas en la base de datos:")
for fila in resumen.itertuples():
    print(f"  {fila.tabla}: {fila.filas:,} filas x {fila.columnas} columnas")

# Índices creados para las uniones por TransactionID y los filtros por tiempo
indices = conn.execute("SELECT name FROM sqlite_master WHERE type = 'index' ORDER BY name").fetchall()
print(f"\nÍndices: {[i[0] for i in indices]}")

# Vista previa de transactions
print("\nPrimeras columnas de transactions:")
print(bd.columnas(conn, 'transactions')[:15])

conn.close()

Tablas en la base de datos:
  features_tarjeta: 590,540 filas x 11 columnas
  identity: 144,233 filas x 41 columnas
  transactions: 590,540 filas x 394 columnas

Índices: ['ix_features_tarjeta_id', 'ix_identity_id', 'ix_transactions_dt', 'ix_transactions_id']

Primeras columnas de transactions:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2']
